In [437]:
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math

In [438]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [439]:
ds = load_dataset("stanfordnlp/sst2")

In [440]:
tokenizer = Tokenizer.from_file(
    f"results/test.json"
)
PAD_TOKEN = "[PAD]"
PAD_ID = tokenizer.token_to_id("[PAD]")
print(f"PAD_ID: {PAD_ID}")

PAD_ID: 1847


In [441]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        # Encode all texts at once for speed
        encodings = tokenizer.encode_batch(texts)
        self.input_ids = [enc.ids for enc in encodings]
        
        # Store labels (assumed to be integers like 0, 1, 2)
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        # Return a paired dictionary for this specific index
        return {
            "input_ids": torch.tensor(self.input_ids[idx], dtype=torch.long),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }


# 3. Create an updated data collator that keeps them paired
def classification_collate_fn(batch):
    # Separate the list of dicts into distinct lists of input_ids and labels
    input_ids_list = [item["input_ids"] for item in batch]
    labels_list = [item["label"] for item in batch]
    
    # 1. Dynamically pad only the input IDs
    padded_input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids_list, 
        batch_first=True, 
        padding_value=PAD_ID
    )
    
    # 2. Stack the labels into a single 1D tensor
    stacked_labels = torch.stack(labels_list)
    
    # Return them paired back up in a batch dictionary
    return padded_input_ids, stacked_labels

In [442]:
dataset = TextClassificationDataset(list(ds["train"]["sentence"]), list(ds["train"]["label"]), tokenizer)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, collate_fn=classification_collate_fn)

In [443]:
class EncoderModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, pad_idx = 0, function = nn.ReLU()):
        super(EncoderModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.ln = nn.LayerNorm(embedding_dim)
        self.hidden_ln = nn.LayerNorm([hidden_dim, hidden_dim])
        self.projection_0 = nn.Linear(embedding_dim, hidden_dim)
        self.projection_1 = nn.Linear(embedding_dim, hidden_dim)
        self.projection_final = nn.Linear(hidden_dim * hidden_dim, output_dim)

        self.function = function

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        embedded = self.ln(embedded)
        a = self.projection_0(embedded)
        b = self.projection_1(embedded)
        output = (a.transpose(-2, -1) @ b) / math.sqrt(input_ids.size(-1))
        output = self.function(output)
        output = self.hidden_ln(output)
        output = self.projection_final(output.view(output.size(0), -1))
        return output

In [444]:
class SkipConnection(nn.Module):
    def __init__(self, in_features, out_features):
        super(SkipConnection, self).__init__()
        self.module = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.ReLU(),
            nn.Linear(in_features, out_features)
        )
        self.skip = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.skip(x) + self.module(x)

In [445]:
model = nn.Sequential(
    EncoderModel(
        vocab_size=tokenizer.get_vocab_size(),
        embedding_dim=512,
        hidden_dim=128,
        output_dim=512,
        pad_idx=PAD_ID
    ),
    SkipConnection(512, 256),
    nn.ReLU(),
    SkipConnection(256, 1),
    nn.Sigmoid()
).to(device)

In [446]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
criterion = nn.BCEWithLogitsLoss()

In [447]:
val_dataset = TextClassificationDataset(list(ds["validation"]["sentence"]), list(ds["validation"]["label"]), tokenizer)
val_dataloader = DataLoader(val_dataset, batch_size=128, shuffle=False, collate_fn=classification_collate_fn)

In [448]:
# --- Voorbereiding (voorbeeld van loss functie) ---
# We gebruiken BCEWithLogitsLoss omdat dit numeriek stabieler is dan handmatige Sigmoid + BCELoss
def evaluate_model(model, val_loader, criterion):
    # 1. Zet het model in evaluatie-modus
    model.eval()
    model.to(device)
    
    total_val_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    # 2. Schakel gradient-berekening uit (bespaart geheugen en rekenkracht)
    with torch.no_grad():
        for batch in val_loader:
            # Data naar de juiste device (CPU of CUDA/GPU) sturen
            input_ids, labels = batch
            input_ids, labels = input_ids.to(device), labels.to(device)
            
            # 3. Forward pass
            # (Dit gaat ervan uit dat je model ruwe logits teruggeeft, niet de ge-sigmoidte kansen)
            outputs = model(input_ids) 
            outputs = outputs.squeeze()  # Vorm aanpassen van [batch_size, 1] naar [batch_size] indien nodig
            
            # 4. Bereken de loss voor deze batch
            loss = criterion(outputs, labels.float())
            total_val_loss += loss.item() * len(labels) # Vermenigvuldigen met batch-grootte voor gewogen gemiddelde
            
            # 5. Bereken de accuracy
            # Aangezien we logits gebruiken: een logit > 0 betekent een voorspelde kans > 0.5 (klasse 1)
            predictions = (outputs > 0.5).long()
            
            # Tel hoeveel voorspellingen exact overeenkomen met de echte labels
            correct_predictions += (predictions == labels.long()).sum().item()
            total_samples += len(labels)
            
    # 6. Bereken de totale gemiddelden over de hele validatieset
    average_val_loss = total_val_loss / total_samples
    accuracy = correct_predictions / total_samples
    
    return average_val_loss, accuracy

# --- Hoe je dit aanroept in je main training loop ---
# device = "cuda" if torch.cuda.is_available() else "cpu"
# val_loss, val_acc = evaluate_model(my_model, val_loader, criterion, device)
# print(f"Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc * 100:.2f}%")


In [449]:
Epochs = 10
for epoch in range(Epochs):
    model.train()
    total_loss = 0
    total_samples = 0
    for batch in dataloader:
        input_ids, labels = batch
        input_ids, labels = input_ids.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids)
        loss = criterion(outputs.squeeze(), labels.float())
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(labels)  # Vermenigvuldig met batch-grootte voor gewogen gemiddelde
        total_samples += len(labels)
    
    avg_loss = total_loss / total_samples
    val_loss, val_acc = evaluate_model(model, val_dataloader, criterion)
    print(f"Epoch {epoch+1}/{Epochs} | Training Loss: {avg_loss:.4f} | Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc * 100:.2f}%")

Epoch 1/10 | Training Loss: 0.6352 | Validation Loss: 0.6184 | Validation Accuracy: 73.28%
Epoch 2/10 | Training Loss: 0.5781 | Validation Loss: 0.6127 | Validation Accuracy: 73.28%
Epoch 3/10 | Training Loss: 0.5684 | Validation Loss: 0.6110 | Validation Accuracy: 75.92%
Epoch 4/10 | Training Loss: 0.5669 | Validation Loss: 0.6160 | Validation Accuracy: 72.48%
Epoch 5/10 | Training Loss: 0.5672 | Validation Loss: 0.6195 | Validation Accuracy: 71.44%
Epoch 6/10 | Training Loss: 0.6530 | Validation Loss: 0.8010 | Validation Accuracy: 51.38%
Epoch 7/10 | Training Loss: 0.6127 | Validation Loss: 0.6535 | Validation Accuracy: 71.22%
Epoch 8/10 | Training Loss: 0.6423 | Validation Loss: 0.6604 | Validation Accuracy: 70.30%
Epoch 9/10 | Training Loss: 0.6499 | Validation Loss: 0.6951 | Validation Accuracy: 62.84%
Epoch 10/10 | Training Loss: 0.7084 | Validation Loss: 0.7637 | Validation Accuracy: 56.54%


In [450]:
model2 = nn.Sequential(
    EncoderModel(
        vocab_size=tokenizer.get_vocab_size(),
        embedding_dim=512,
        hidden_dim=128,
        output_dim=512,
        pad_idx=PAD_ID
    ),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, 1),
    nn.Sigmoid()
).to(device)

In [451]:
optimizer2 = torch.optim.AdamW(model2.parameters(), lr=5e-4)

In [452]:
Epochs = 10
for epoch in range(Epochs):
    model2.train()
    total_loss = 0
    total_samples = 0
    for batch in dataloader:
        input_ids, labels = batch
        input_ids, labels = input_ids.to(device), labels.to(device)
        
        optimizer2.zero_grad()
        outputs = model2(input_ids)
        loss = criterion(outputs.squeeze(), labels.float())
        loss.backward()
        optimizer2.step()
        
        total_loss += loss.item() * len(labels)  # Vermenigvuldig met batch-grootte voor gewogen gemiddelde
        total_samples += len(labels)
    
    avg_loss = total_loss / total_samples
    val_loss, val_acc = evaluate_model(model2, val_dataloader, criterion)
    print(f"Epoch {epoch+1}/{Epochs} | Training Loss: {avg_loss:.4f} | Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc * 100:.2f}%")

Epoch 1/10 | Training Loss: 0.6031 | Validation Loss: 0.6111 | Validation Accuracy: 77.06%
Epoch 2/10 | Training Loss: 0.5611 | Validation Loss: 0.6042 | Validation Accuracy: 77.06%
Epoch 3/10 | Training Loss: 0.5505 | Validation Loss: 0.6023 | Validation Accuracy: 77.52%
Epoch 4/10 | Training Loss: 0.5445 | Validation Loss: 0.5976 | Validation Accuracy: 78.78%
Epoch 5/10 | Training Loss: 0.5429 | Validation Loss: 0.6023 | Validation Accuracy: 76.72%
Epoch 6/10 | Training Loss: 0.5398 | Validation Loss: 0.6075 | Validation Accuracy: 77.87%
Epoch 7/10 | Training Loss: 0.5390 | Validation Loss: 0.6062 | Validation Accuracy: 78.21%
Epoch 8/10 | Training Loss: 0.5412 | Validation Loss: 0.6064 | Validation Accuracy: 77.06%
Epoch 9/10 | Training Loss: 0.5370 | Validation Loss: 0.6070 | Validation Accuracy: 75.00%
Epoch 10/10 | Training Loss: 0.5345 | Validation Loss: 0.5944 | Validation Accuracy: 77.64%


In [453]:
print("Evaluating model on validation set...")
val_loss, val_acc = evaluate_model(model, val_dataloader, criterion)
print(f"Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc * 100:.2f}%")

Evaluating model on validation set...
Validation Loss: 0.7637 | Validation Accuracy: 56.54%
